# Pixasonics temporal frame-by-frame demo

This notebook shows one way to emulate video playback by preloading frames and updating the Pixasonics app one frame at a time.

The key idea is to convert each video frame to a NumPy array and call `app.load_image_data(...)` for the current frame.

In [1]:
import sys
print(sys.executable)

c:\Python314\python.exe


In [2]:
import os
import time

import cv2
import numpy as np

from pixasonics.core import App, Mapper
from pixasonics.features import MeanChannelValue
from pixasonics.synths import Theremin

In [3]:
def load_video_frames(video_path, max_frames=None):
    frames = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f'Could not open video: {video_path}')

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if max_frames is not None and len(frames) >= max_frames:
            break

    cap.release()
    return frames

In [4]:
VIDEO_PATH = os.path.join('.', 'example_video.mp4')

if not os.path.exists(VIDEO_PATH):
    print(f'No file found at {VIDEO_PATH}. Edit VIDEO_PATH to point to your video, or download one into this folder.')
    VIDEO_PATH = None

In [ ]:
app = App()
app.audio = True
app.master_volume = 7

# Important: Pixasonics expects a valid image in the app before attaching a feature.
# A blank/empty image causes the IndexError you were seeing.
base_frame = np.zeros((64, 64, 3), dtype=np.uint8)
base_frame[:, :, 0] = 255  # bright red image as a valid starter frame
app.load_image_data(base_frame, refresh_features=True)

mean_red = MeanChannelValue(filter_channels=0, name='MeanRed')
app.attach(mean_red)

theremin = Theremin(name='MySine')
app.attach(theremin)

red2freq = Mapper(mean_red, theremin['frequency'], exponent=2, name='Red2Freq')
app.attach(red2freq)

Box(children=(Box(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='…

IndexError: too many indices for array: array is 3-dimensional, but 4 were indexed

In [ ]:
if VIDEO_PATH is not None:
    frames = load_video_frames(VIDEO_PATH, max_frames=120)
    print(f'Loaded {len(frames)} frames from {VIDEO_PATH}')

    for i, frame in enumerate(frames):
        app.load_image_data(frame, refresh_features=True)
        print(f'Playing frame {i + 1}/{len(frames)}')
        time.sleep(0.05)
else:
    print('Add a valid video file to VIDEO_PATH and rerun this cell.')

## Notes

- This is a practical way to mimic a video stream with Pixasonics.
- `load_image_data(...)` accepts a NumPy array, so each frame can be fed in one at a time.
- If you want a real-time playback loop, replace the `time.sleep(0.05)` with your preferred frame timing.
- For a smoother control signal, you can also extract one feature per frame and map it into the synth in a loop.